# Loading Structured Data: Mastering JSON with LangChain Loaders

In advanced Retrieval-Augmented Generation (RAG) systems, the knowledge base rarely consists of simple, monolithic text documents. Instead, real-world data is stored in highly structured formats like JSON, XML, or database records. The ability to ingest and correctly interpret this structured data is a critical skill for any developer building production-grade AI applications. This notebook focuses on mastering the `JSONLoader`, demonstrating how to move beyond basic file reading to sophisticated data parsing.

We will learn techniques such as using `jq_schema` to navigate complex JSON arrays (e.g., extracting every item from a root array like `.products[]`) and, most importantly, implementing custom metadata extraction via `metadata_func`. This function allows us to programmatically define which fields—like product names, categories, or prices—should be treated as rich, searchable metadata attached to the document chunk. By meticulously cleaning and enriching this metadata, we ensure that our vector store contains not just text content, but also highly structured context that can guide sophisticated retrieval logic in a LangGraph workflow.

Understanding how to transform raw JSON into semantically rich `Document` objects is foundational for building robust RAG pipelines. If the initial loading step fails to correctly identify and separate key entities (like product IDs or prices) from the main description text, the entire system's ability to answer precise questions will be compromised. This deep dive equips you with the tools necessary to handle complex data schemas, making your knowledge base reliable, searchable, and ready for advanced reasoning tasks.

### Learning Objectives
*   **Structured Data Ingestion:** Understand how to use `JSONLoader` to load content from JSON files, moving beyond simple text loading.
*   **Schema Parsing (`jq_schema`):** Learn to utilize the `jq_schema` parameter to efficiently parse and iterate over complex array structures within a single JSON file (e.g., extracting multiple records from an array).
*   **Custom Metadata Extraction:** Master the use of `metadata_func` to programmatically define, extract, clean, and enrich document metadata using data fields that are not part of the main text content.
*   **Data Cleaning:** Implement best practices for cleaning loaded documents by explicitly deleting unwanted or redundant metadata keys (e.g., removing sequence numbers).


In [1]:
from langchain_community.document_loaders.json_loader import JSONLoader
from pathlib import Path
from pprint import pp

### File Path Setup and Validation

This cell initializes a `Path` object pointing to the JSON knowledge source file. The subsequent call to `.exists()` is crucial for validating that the specified file path is accessible before attempting to load or process the data, preventing runtime errors.


In [2]:
# create the path for json file

# Initialize a Path object using the relative directory structure.
file_path = Path("../knowledge-source/apparels.json")

# Check if the file specified by file_path actually exists in the filesystem.
file_path.exists()


True

### Metadata Transformation Function

This function, `metadata_func`, is a crucial preprocessing step used to standardize and clean the metadata associated with loaded records. It extracts specific fields (like product name, category, and price) from the raw record dictionary and updates the provided metadata dictionary while explicitly removing unnecessary keys such as `seq_num`.


In [21]:
def metadata_func(record: dict, metadata: dict) -> dict:
    # Extract core business data fields from the source record
    metadata["product_name"] = record["productName"]
    metadata["category"] = record["category"]
    metadata["price"] = record["price"]
    # Delete sequence number (seq_num) as it is not useful for retrieval or LLM context
    del metadata["seq_num"]
    return metadata


### JSONLoader Initialization

This cell initializes the `JSONLoader`, which is responsible for reading structured data from a local JSON file. It uses `jq_schema` to selectively extract an array of product objects (`.products[]`) and specifies that the main content should be taken from the `Description` field, while custom metadata is added via `metadata_func`.


In [22]:
# form the loader

loader = JSONLoader(file_path=file_path.as_posix(), # Specify the path to the local JSON file.
                    jq_schema=".products[]", # Use jq to extract an array of product objects from the 'products' key.
                    content_key="Description", # Designate the 'Description' field as the primary content for the loader.
                    metadata_func=metadata_func) # Pass a function to generate rich metadata for each loaded document.


### Document Loading

This cell executes the document loading process. It calls the `load()` method on the initialized `loader` object (which is assumed to be a specialized loader class) to ingest all necessary documents from the source, storing them in the `documents` variable.


In [23]:
# documents

# Load all documents using the pre-configured loader instance.
documents = loader.load()


This loop iterates through the `documents` list (which contains loaded document objects, likely from a library like LlamaIndex or LangChain). It prints the textual content (`page_content`) of each document object to standard output. This step is crucial for debugging and verifying that the documents were correctly loaded and contain the expected text before proceeding with advanced RAG operations.


In [24]:
for doc in documents:
    print(doc.page_content, end="\n\n")

Oversize-fit coat made of a viscose blend fabric. Notch lapel collar and long sleeves with buttoned cuffs.

Trench coat made of technical fabric with a velvety finish. Notch lapel collar and long sleeves.

Parka made of technical fabric, padded on inside. High neck with a hood and long sleeves with elasticated cuffs.

The first Vibrant Leather for le benjamin with its fruity hint of pineapple that gives it modernity, accompanied by cedar notes, adding a woody touch of rejuvenation.

Multicoloured backpack. Soft construction. Featuring one main zip pocket and a medium-sized zip pocket on the inside.



This cell accesses the metadata associated with the first document in the `documents` list. Metadata typically contains crucial contextual information (like source file name, page number, or creation date) that is vital for advanced RAG systems to improve grounding and traceability.


In [25]:
documents[0].metadata # Accesses the metadata dictionary of the first document object in the 'documents' list.


{'source': 'D:\\rag-document-loaders\\knowledge-source\\apparels.json',
 'product_name': 'PINSTRIPE COAT',
 'category': 'Men Cloths',
 'price': 4900}

### Code Explanation

This cell simply calculates and displays the total number of documents loaded into the `documents` variable. This is a crucial step for debugging and verifying that the document loading process (e.g., using `json_loader`) successfully retrieved all expected files.


In [26]:
len(documents) # Calculates the length of the 'documents' list/variable, which represents the total count of loaded documents.


5